# Demo — Lineage as a Blast-Radius Map

Lineage answers two operational questions: if this dataset breaks, **what downstream is
affected?** (`trace_downstream` / `simulate_breach`) and for a regulated output, **what sources
feed it?** (`trace_upstream`). This walkthrough reads the Trailhead lineage graph, traces the
shipment-to-carbon pipeline in both directions, and specifies one monitoring signal; the exercise
asks you to trace a customer PII breach, verify the CSRD report's full upstream provenance, and
specify the platform monitoring table.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
display(gc.lineage.events())
print("\nMermaid (paste into any Mermaid renderer):\n")
print(gc.lineage.to_mermaid())

## Trace downstream (blast radius) and upstream (provenance)
Use `trace_downstream()` and `simulate_breach()` to see what a defect in `shipment` contaminates,
and `trace_upstream()` to verify which source tables feed the intermediate `order_carbon` table:

In [ ]:
print("Downstream of shipment:", gc.lineage.trace_downstream("shipment"))
print("Upstream of order_carbon:", gc.lineage.trace_upstream("order_carbon"))

breach = gc.lineage.simulate_breach("shipment")
breach

## Place a monitoring signal at the critical junction
Because a defect at `shipment` flows directly into `order_carbon` and then
`quarterly_sustainability` with no gate in between, we place a monitoring signal on that edge:

| Signal | Source dataset/job | Threshold | Owner | Ties to |
|---|---|---|---|---|
| `shipment_carbon_range` | `shipment` → `build_order_carbon` | `carbon_kg <= 100` (0 rows above max) | `inventory-fulfillment@trailhead.example` | CSRD carbon accuracy |

**Takeaway / your turn:** in the exercise you'll trace a `customer` PII breach downstream with
`trace_downstream()`, trace all upstream sources of `quarterly_sustainability` with
`trace_upstream()`, and specify the full monitoring table across the platform.